**Importing Libraries**

In [35]:
from collections import defaultdict, Counter
import pandas as pd

**Importing the Dataset**

In [3]:
file_path = r"C:\Asia Pacific University\All Module Notes\Semister-5\Text Analysis & Sentimental Analysis\Assignment\CT107-3-3-TXSA - Group Assignment\Data\Data_3.txt"

with open(file_path, 'r') as file:
    content = file.read()
    
print(content)

Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a different book by Danielle </s>


**Parsing Training Corpus and Test Sentence**

In [7]:
# Used for splitting into lines
lines = content.strip().split('\n')

# Extracting training corpus and test sentence
corpus = []
test_sentence = None

in_training = False
for line in lines:
    line = line.strip()
    
    if line.startswith('Training Corpus'):
        in_training = True
        continue
    
    if 'Calculate sentence probability' in line:
        in_training = False
        continue
    
    # Skip empty lines and separator lines
    if not line or line.startswith('~'):
        continue
    
    # Add sentences with <s> markers
    if '<s>' in line:
        if in_training:
            corpus.append(line)
        else:
            test_sentence = line


print("Training Corpus:")
for i, sentence in enumerate(corpus, 1):
    print(f"{i}. {sentence}")

print("\n")

print("Test Sentence:")
print(test_sentence)

Training Corpus:
1. <s> He read a book </s>
2. <s> I read a different book </s>
3. <s> He read a book by Danielle </s>


Test Sentence:
<s> I read a different book by Danielle </s>


**Tokenization of Sentences in Training Corpus**

In [8]:
# Tokenize each sentences into words using simple split
tokenized_corpus = [sentence.split() for sentence in corpus]

print("Tokenized Corpus:")
for i, tokens in enumerate(tokenized_corpus, 1):
    print(f"{i}. {tokens}")

Tokenized Corpus:
1. ['<s>', 'He', 'read', 'a', 'book', '</s>']
2. ['<s>', 'I', 'read', 'a', 'different', 'book', '</s>']
3. ['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']


**Extracting Vocabolaries in Training Corpus(unique words)** 

In [10]:
vocabulary = set()
for tokens in tokenized_corpus:
    vocabulary.update(tokens)

vocabulary = sorted(vocabulary)

print("Vocabulary:")
print(vocabulary)
print(f"\nVocabulary Size (V): {len(vocabulary)}")

Vocabulary:
['</s>', '<s>', 'Danielle', 'He', 'I', 'a', 'book', 'by', 'different', 'read']

Vocabulary Size (V): 10


**Count Unigrams**

In [13]:
unigram_counts = Counter()

for tokens in tokenized_corpus:
    unigram_counts.update(tokens)

print("Unigram Counts:")
for word in vocabulary:
    print(f"C({word:2s}) = {unigram_counts[word]}")

Unigram Counts:
C(</s>) = 3
C(<s>) = 3
C(Danielle) = 1
C(He) = 2
C(I ) = 1
C(a ) = 3
C(book) = 3
C(by) = 1
C(different) = 1
C(read) = 3


**Count Bigrams**

In [16]:
# Count bigrams (words in pairs)
bigram_counts = Counter()

for tokens in tokenized_corpus:
    # Create bigrams from consecutive words
    for i in range(len(tokens) - 1):
        bigram = (tokens[i], tokens[i + 1])
        bigram_counts[bigram] += 1
        
print("Bigram Counts:")
for bigram, count in sorted(bigram_counts.items()):
    print(f"C({bigram[0]:1s}, {bigram[1]:1s}) = {count}")


Bigram Counts:
C(<s>, He) = 2
C(<s>, I) = 1
C(Danielle, </s>) = 1
C(He, read) = 2
C(I, read) = 1
C(a, book) = 2
C(a, different) = 1
C(book, </s>) = 2
C(book, by) = 1
C(by, Danielle) = 1
C(different, book) = 1
C(read, a) = 3


**Preparing Test Sentence**

***Tokenization of the test sentence***

In [18]:
test_tokens = test_sentence.split()

print("Target Sentence:")
print(test_sentence)

print(f"\nTokens: {test_tokens}")
print(f"Number of tokens: {len(test_tokens)}")

Target Sentence:
<s> I read a different book by Danielle </s>

Tokens: ['<s>', 'I', 'read', 'a', 'different', 'book', 'by', 'Danielle', '</s>']
Number of tokens: 9


***Extraction of bigrams from test sentence***

In [ ]:
test_bigrams = []
for i in range(len(test_tokens) - 1):
    test_bigrams.append((test_tokens[i], test_tokens[i + 1]))
    
print("\nBigrams in Test Sentence:")
for i, bigram in enumerate(test_bigrams, 1):
    print(f"{i}. ({bigram[0]}, {bigram[1]})")


Bigrams in Test Sentence:
1. (<s>, I)
2. (I, read)
3. (read, a)
4. (a, different)
5. (different, book)
6. (book, by)
7. (by, Danielle)
8. (Danielle, </s>)


**Calculation of Unsmoothed Bigram Probabilities**

In [23]:
print("UNSMOOTHED BIGRAM MODEL")

unsmoothed_probs = []

for bigram in test_bigrams:
    w_prev, w_curr = bigram
    
    # Get counts
    bigram_count = bigram_counts[bigram]
    unigram_count = unigram_counts[w_prev]
    
    # Calculate probability
    prob = bigram_count / unigram_count
    
    unsmoothed_probs.append(prob)
    
    # Display calculation
    print(f"\nP({w_curr:1s} | {w_prev:1s})")
    
    print(f"  = C({w_prev}, {w_curr}) / C({w_prev})")
    
    print(f"  = {bigram_count} / {unigram_count}")
    
    print(f"  = {bigram_count}/{unigram_count}")
    
    print(f"  = {prob:.4f}")

UNSMOOTHED BIGRAM MODEL

P(I | <s>)
  = C(<s>, I) / C(<s>)
  = 1 / 3
  = 1/3
  = 0.3333

P(read | I)
  = C(I, read) / C(I)
  = 1 / 1
  = 1/1
  = 1.0000

P(a | read)
  = C(read, a) / C(read)
  = 3 / 3
  = 3/3
  = 1.0000

P(different | a)
  = C(a, different) / C(a)
  = 1 / 3
  = 1/3
  = 0.3333

P(book | different)
  = C(different, book) / C(different)
  = 1 / 1
  = 1/1
  = 1.0000

P(by | book)
  = C(book, by) / C(book)
  = 1 / 3
  = 1/3
  = 0.3333

P(Danielle | by)
  = C(by, Danielle) / C(by)
  = 1 / 1
  = 1/1
  = 1.0000

P(</s> | Danielle)
  = C(Danielle, </s>) / C(Danielle)
  = 1 / 1
  = 1/1
  = 1.0000


**Calculate of Total Unsmoothed Sentence Probability**

In [24]:
print("UNSMOOTHED SENTENCE PROBABILITY")

# Calculate sentence probability (product of all bigram probabilities)
sentence_prob_unsmoothed = 1.0
for prob in unsmoothed_probs:
    sentence_prob_unsmoothed *= prob

# Show calculation step by step
print("P(sentence) = P(I|<s>) × P(read|I) × P(a|read) × P(different|a)")
print("            × P(book|different) × P(by|book) × P(Danielle|by) × P(</s>|Danielle)")
print()

# Show as fractions
fraction_str = " × ".join([f"({bigram_counts[b]}/{unigram_counts[b[0]]})" for b in test_bigrams])
print(f"            = {fraction_str}")
print()

# Show as decimals
decimal_str = " × ".join([f"{p:.4f}" for p in unsmoothed_probs])
print(f"            = {decimal_str}")
print()

# Final result
print(f"            = {sentence_prob_unsmoothed:.6f}")
print(f"            = {sentence_prob_unsmoothed:.4f}")
print(f"            = 1/27")
print(f"            = {sentence_prob_unsmoothed * 100:.2f}%")

UNSMOOTHED SENTENCE PROBABILITY
P(sentence) = P(I|<s>) × P(read|I) × P(a|read) × P(different|a)
            × P(book|different) × P(by|book) × P(Danielle|by) × P(</s>|Danielle)

            = (1/3) × (1/1) × (3/3) × (1/3) × (1/1) × (1/3) × (1/1) × (1/1)

            = 0.3333 × 1.0000 × 1.0000 × 0.3333 × 1.0000 × 0.3333 × 1.0000 × 1.0000

            = 0.037037
            = 0.0370
            = 1/27
            = 3.70%


**Calculation of Smoothed  Bigram Probabilities (Add-1)**

In [27]:
print("SMOOTHED BIGRAM MODEL (Add-1 Laplace Smoothing)")
print("=" * 70)
print("Formula: P(wₙ|wₙ₋₁) = (C(wₙ₋₁wₙ) + 1) / (C(wₙ₋₁) + V)")
print(f"Where V = {len(vocabulary)}")
print("=" * 70)

V = len(vocabulary)
smoothed_probs = []

for bigram in test_bigrams:
    w_prev, w_curr = bigram
    
    # Get counts
    bigram_count = bigram_counts[bigram]
    unigram_count = unigram_counts[w_prev]
    
    # Calculate smoothed probability
    prob = (bigram_count + 1) / (unigram_count + V)
    
    smoothed_probs.append(prob)
    
    # Display calculation
    print(f"\nP({w_curr:1s} | {w_prev:1s})")
    print(f"  = (C({w_prev}, {w_curr}) + 1) / (C({w_prev}) + V)")
    print(f"  = ({bigram_count} + 1) / ({unigram_count} + {V})")
    print(f"  = {bigram_count + 1} / {unigram_count + V}")
    print(f"  = {bigram_count + 1}/{unigram_count + V}")
    print(f"  = {prob:.4f}")

SMOOTHED BIGRAM MODEL (Add-1 Laplace Smoothing)
Formula: P(wₙ|wₙ₋₁) = (C(wₙ₋₁wₙ) + 1) / (C(wₙ₋₁) + V)
Where V = 10

P(I | <s>)
  = (C(<s>, I) + 1) / (C(<s>) + V)
  = (1 + 1) / (3 + 10)
  = 2 / 13
  = 2/13
  = 0.1538

P(read | I)
  = (C(I, read) + 1) / (C(I) + V)
  = (1 + 1) / (1 + 10)
  = 2 / 11
  = 2/11
  = 0.1818

P(a | read)
  = (C(read, a) + 1) / (C(read) + V)
  = (3 + 1) / (3 + 10)
  = 4 / 13
  = 4/13
  = 0.3077

P(different | a)
  = (C(a, different) + 1) / (C(a) + V)
  = (1 + 1) / (3 + 10)
  = 2 / 13
  = 2/13
  = 0.1538

P(book | different)
  = (C(different, book) + 1) / (C(different) + V)
  = (1 + 1) / (1 + 10)
  = 2 / 11
  = 2/11
  = 0.1818

P(by | book)
  = (C(book, by) + 1) / (C(book) + V)
  = (1 + 1) / (3 + 10)
  = 2 / 13
  = 2/13
  = 0.1538

P(Danielle | by)
  = (C(by, Danielle) + 1) / (C(by) + V)
  = (1 + 1) / (1 + 10)
  = 2 / 11
  = 2/11
  = 0.1818

P(</s> | Danielle)
  = (C(Danielle, </s>) + 1) / (C(Danielle) + V)
  = (1 + 1) / (1 + 10)
  = 2 / 11
  = 2/11
  = 0.1818


**Calculate Smoothed Sentence Probability**

In [43]:
print("=" * 70)
print("SMOOTHED SENTENCE PROBABILITY")
print("=" * 70)

# Calculate sentence probability (product of all bigram probabilities)
sentence_prob_smoothed = 1.0
for prob in smoothed_probs:
    sentence_prob_smoothed *= prob

# Show calculation step by step
print("P(sentence) = P(I|<s>) × P(read|I) × P(a|read) × P(different|a)")
print("            × P(book|different) × P(by|book) × P(Danielle|by) × P(</s>|Danielle)")
print()

# Show as fractions
fraction_parts = []
for b in test_bigrams:
    bc = bigram_counts[b]
    uc = unigram_counts[b[0]]
    fraction_parts.append(f"({bc + 1}/{uc + V})")
fraction_str = " × ".join(fraction_parts)
print(f"            = {fraction_str}")
print()

# Show as decimals
decimal_str = " × ".join([f"{p:.4f}" for p in smoothed_probs])
print(f"            = {decimal_str}")
print()

# Final result
print(f"            = {sentence_prob_smoothed:.10f}")
print(f"            = {sentence_prob_smoothed:.7f}")
print(f"            = {sentence_prob_smoothed:.2e}")

SMOOTHED SENTENCE PROBABILITY
P(sentence) = P(I|<s>) × P(read|I) × P(a|read) × P(different|a)
            × P(book|different) × P(by|book) × P(Danielle|by) × P(</s>|Danielle)

            = (2/13) × (2/11) × (4/13) × (2/13) × (2/11) × (2/13) × (2/11) × (2/11)

            = 0.1538 × 0.1818 × 0.3077 × 0.1538 × 0.1818 × 0.1538 × 0.1818 × 0.1818

            = 0.0000012244
            = 0.0000012
            = 1.22e-06


**Comparing Results**

In [40]:
print("=" * 90)
print("COMPARISON OF RESULTS".center(90))
print("=" * 90)
print()

# Create comparison table
comparison_data = {
    'Model': ['Unsmoothed Bigram', 'Smoothed Bigram (Add-1)'],
    'Decimal': [
        f"{sentence_prob_unsmoothed:.6f}", 
        f"{sentence_prob_smoothed:.10f}"
    ],
    'Scientific': [
        f"{sentence_prob_unsmoothed:.2e}",
        f"{sentence_prob_smoothed:.2e}"
    ],
    'Percentage': [
        f"{sentence_prob_unsmoothed * 100:.2f}%",
        f"{sentence_prob_smoothed * 100:.6f}%"
    ]
}

df_comparison = pd.DataFrame(comparison_data)

# Display as table 
display(df_comparison)  

print()
print("=" * 90)
print("ANALYSIS".center(90))
print("=" * 90)

ratio = sentence_prob_unsmoothed / sentence_prob_smoothed

print(f"Probability Ratio:")
print(f"   Unsmoothed is {ratio:,.0f}x higher than smoothed\n")

print(f" Why smoothed probability is lower:")
print(f"   • Add-1 smoothing redistributes probability mass to ALL possible bigrams")
print(f"   • Added 1 to every bigram count (creating {len(vocabulary)} new possible bigrams per word)")
print(f"   • Prevents zero probabilities but reduces probabilities for observed sequences\n")

print(f"When to use each model:")
print(f"   • Unsmoothed: Good when test data is similar to training data")
print(f"   • Smoothed:   Better for real-world applications (handles unseen bigrams)")
print()
print("=" * 90)

                                  COMPARISON OF RESULTS                                   



,Model,Decimal,Scientific,Percentage
0,Unsmoothed Bigram,0.037037,3.70e-02,3.70%
1,Smoothed Bigram (Add-1),0.0000012244,1.22e-06,0.000122%



                                         ANALYSIS                                         
Probability Ratio:
   Unsmoothed is 30,249x higher than smoothed

 Why smoothed probability is lower:
   • Add-1 smoothing redistributes probability mass to ALL possible bigrams
   • Added 1 to every bigram count (creating 10 new possible bigrams per word)
   • Prevents zero probabilities but reduces probabilities for observed sequences

When to use each model:
   • Unsmoothed: Good when test data is similar to training data
   • Smoothed:   Better for real-world applications (handles unseen bigrams)



**Individual Bigram Probabilities**

In [42]:
print("=" * 100)
print("Individual Bigram Probabilities".center(100))
print("=" * 100)
print()

# Build verification data
verification_data = []
for bigram in test_bigrams:
    w_prev, w_curr = bigram
    bc = bigram_counts[bigram]
    uc = unigram_counts[w_prev]
    
    unsmoothed_val = bc / uc
    smoothed_val = (bc + 1) / (uc + V)
    difference = unsmoothed_val - smoothed_val
    
    verification_data.append({
        'Bigram': f"({w_prev}, {w_curr})",
        'Unsmoothed': f"{bc}/{uc} = {unsmoothed_val:.4f}",
        'Smoothed': f"{bc+1}/{uc+V} = {smoothed_val:.4f}",
        'Difference': f"{difference:.4f}"
    })

df_verification = pd.DataFrame(verification_data)

# Display as HTML table
display(df_verification)  

                                  Individual Bigram Probabilities                                   



,Bigram,Unsmoothed,Smoothed,Difference
0,"(<s>, I)",1/3 = 0.3333,2/13 = 0.1538,0.1795
1,"(I, read)",1/1 = 1.0000,2/11 = 0.1818,0.8182
2,"(read, a)",3/3 = 1.0000,4/13 = 0.3077,0.6923
3,"(a, different)",1/3 = 0.3333,2/13 = 0.1538,0.1795
4,"(different, book)",1/1 = 1.0000,2/11 = 0.1818,0.8182
5,"(book, by)",1/3 = 0.3333,2/13 = 0.1538,0.1795
6,"(by, Danielle)",1/1 = 1.0000,2/11 = 0.1818,0.8182
7,"(Danielle, </s>)",1/1 = 1.0000,2/11 = 0.1818,0.8182
